In [ ]:
import Bio.SeqIO
import pandas as pd
import gzip
import plotly.express as px
import plotly.graph_objects as go

import sys
sys.path.append('../')
import plotting

In [ ]:
dfs = []
for exp in ('GCall', 'GCfix'):
    for source in ('R1', 'unmapped'):
        for i in range(1, 6+1):
            sequences = Bio.SeqIO.parse(gzip.open(f"../data/internal_datasets/{exp}/PCR{i}/{source}.fq.gz", "rt"), "fastq")
            # get average quality per read into df
            dfs.append(pd.DataFrame({
                "quality": [sum(seq.letter_annotations["phred_quality"]) / len(seq) for seq in sequences],
                "exp": exp,
                "source": source,
                "PCR": i,
            }))

df = pd.concat(dfs).reset_index(drop=True)

In [ ]:
plotdf = df.groupby(["exp", "source", "PCR"]).agg(
    mean_quality=("quality", "mean"),
    std_quality=("quality", "std"),
).reset_index()
plotdf['n_cycles'] = 15*plotdf['PCR']

fig = px.bar(
    plotdf,
    x="n_cycles",
    y="mean_quality",
    color="source",
    barmode="group",
    error_y="std_quality",
    facet_col="exp",
)

fig.update_yaxes(
    title="Mean Phred quality score",
    row=1,
    col=1,
)

fig.update_yaxes(
    dtick=10,
    minor_dtick=5,
)

fig.update_xaxes(
    title="PCR cycles",
    dtick=15
)

fig.update_layout(
    width=680,
    height=200,
    margin=dict(l=0, r=5, t=25, b=0),
    showlegend=False
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig = plotting.standardize_plot(fig)
fig.show()
fig.write_image("SI_figure_sequencing_quality/PCR_cycles.svg")